In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import inspect

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore")


file_path = "/content/adult.csv"


target_column = "income"

df = pd.read_csv(file_path)

print("First 5 rows:")
print(df.head())

print("\nDataset Shape:", df.shape)

df.columns = df.columns.astype(str).str.strip()

if target_column not in df.columns:
    raise ValueError(
        f"Target column '{target_column}' not found. "
        f"Available columns: {df.columns.tolist()}"
    )

df = df.drop_duplicates()
df = df.dropna(subset=[target_column])


X = df.drop(columns=[target_column]).copy()
y = df[target_column].copy()


print("\nTarget Distribution:")
print(y.value_counts())


label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y.astype(str))

print("\nNumber of Classes:", len(np.unique(y)))


if len(np.unique(y)) < 2:
    raise ValueError("Target must contain at least two classes.")


if len(np.unique(y)) != 2:
    raise ValueError(
        "This version supports binary classification only. "
        "For multiclass datasets, ROC analysis needs a different approach."
    )


numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("\nNumerical Features:", numeric_features)
print("Categorical Features:", categorical_features)


numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)


base_tree = DecisionTreeClassifier(random_state=42)

bagging_args = {
    "n_estimators": 50,
    "bootstrap": True,
    "n_jobs": -1,
    "random_state": 42
}

if "estimator" in inspect.signature(BaggingClassifier).parameters:
    bagging_model = BaggingClassifier(
        estimator=base_tree,
        **bagging_args
    )
else:
    bagging_model = BaggingClassifier(
        base_estimator=base_tree,
        **bagging_args
    )

bagging = Pipeline([
    ("preprocessing", preprocessor),
    ("model", bagging_model)
])

bagging.fit(X_train, y_train)

print("\nBagging Model Trained Successfully!")


boosting_model = AdaBoostClassifier(
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

boosting = Pipeline([
    ("preprocessing", preprocessor),
    ("model", boosting_model)
])

boosting.fit(X_train, y_train)

print("AdaBoost Model Trained Successfully!")

models = {
    "Bagging": bagging,
    "AdaBoost": boosting
}

results = []

for name, model in models.items():

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

comparison = pd.DataFrame(results)

print("\nMODEL PERFORMANCE COMPARISON:")
print(comparison.round(4).to_string(index=False))

for name, model in models.items():

    print("\n", name, "CLASSIFICATION REPORT")

    print(classification_report(
        y_test,
        model.predict(X_test),
        zero_division=0
    ))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, model) in zip(axes, models.items()):

    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test,
        ax=ax,
        cmap="Blues",
        colorbar=False
    )

    ax.set_title(name)

plt.tight_layout()
plt.show()

# 13. ROC CURVE
plt.figure(figsize=(8, 6))

for name, model in models.items():

    probabilities = model.predict_proba(X_test)[:, 1]

    fpr, tpr, thresholds = roc_curve(
        y_test,
        probabilities
    )

    auc_score = roc_auc_score(
        y_test,
        probabilities
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {auc_score:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="black",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Bagging vs AdaBoost")
plt.legend()
plt.grid()
plt.show()

# 14. ACCURACY COMPARISON
comparison.set_index("Model")["Accuracy"].plot(
    kind="bar",
    figsize=(7, 5)
)

plt.title("Bagging vs Boosting Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()

print("\nEXPERIMENT COMPLETED SUCCESSFULLY!")


First 5 rows:
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income  
0          4356              40  United-States  <=50K  
1          4356              18 